# Rede neural convolucional sem transformada Wavelet no pipeline 


## 1. Dataset - MiniMIAS

### 1.1 Rodar localmente o programa

- Para isso é necessário baixar e extrair o dataset para a máquina local. 

**Link:** https://www.repository.cam.ac.uk/items/b6a97f0c-3b9b-40ad-8f18-3d121eef1459

- Baixe o dataset em Files;
- Após extrair em uma pasta os arquivos. Comando via terminal: _unzip MIAS-DB.zip_

In [5]:
import os

pasta = "dataset-mias/"

arquivos = os.listdir(pasta)
contador = 0

for arquivo in arquivos:
    if ".pgm" in arquivo:
        contador += 1

print(contador)

322


### 1.2 Classificando as imagnes

In [1]:
import os
import random
import shutil
from PyPDF2 import PdfReader

pasta_imagens = "dataset-mias/"

com_nodulo = []
sem_nodulo = []

# LER PDF
reader = PdfReader("dataset-mias/00README.pdf")
linhas = []

for pagina in reader.pages:
    texto = pagina.extract_text()
    linhas.extend(texto.split("\n"))

# PROCESSAR
for linha in linhas:
    if "mdb" in linha:
        partes = linha.split()
        nome = partes[0] + ".pgm"
        
        if "NORM" in linha:
            sem_nodulo.append(nome)
        else:
            com_nodulo.append(nome)

# DIVIDIR
def dividir(lista, pasta_train, pasta_test):
    random.shuffle(lista)
    corte = int(0.7 * len(lista))
    
    for nome in lista[:corte]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_train, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)
    
    for nome in lista[corte:]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_test, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)

# PASTAS
os.makedirs("dataset/train/com_nodulo", exist_ok=True)
os.makedirs("dataset/train/sem_nodulo", exist_ok=True)
os.makedirs("dataset/test/com_nodulo", exist_ok=True)
os.makedirs("dataset/test/sem_nodulo", exist_ok=True)

# EXECUTAR
dividir(com_nodulo, "dataset/train/com_nodulo", "dataset/test/com_nodulo")
dividir(sem_nodulo, "dataset/train/sem_nodulo", "dataset/test/sem_nodulo")

print("Finalizado!")

Finalizado!


### 1.3 Carregando dataset para CNN

#### Data Augmentation (aumento de dados)

Agora para o modelo 4 vamos aumentar nosso dataset simplesmente efetuando rotações nas imagnes mamográficas.

Dessa forma, vamos aplicar Data Augmentation nos dados de treino e manter os dados de teste apenas redimensionado com a imagem limpa.

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Transformação para TREINO 
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    # --- Data Augmentation Início ---
    transforms.RandomHorizontalFlip(p=0.5), # Gira a imagem às vezes
    transforms.RandomRotation(15),          # Gira levemente (até 15 graus)
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Move um pouco
    # --- Data Augmentation Fim ---
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))    # Ajuda a rede a convergir mais rápido
])

# 2. Transformação para TESTE 
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 3. Aplicar cada transformação no seu devido lugar
train_dataset = datasets.ImageFolder(
    root="dataset/train",
    transform=train_transform 
)

test_dataset = datasets.ImageFolder(
    root="dataset/test",
    transform=test_transform
)

# Cria DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"Classes: {train_dataset.classes}")

Classes: ['com_nodulo', 'sem_nodulo']


## 2. Criando uma camada Wavelet usando PyTorch

#### WaveletLayer Treinável

Agora vamos permiti que a rede aprenda e melhore os pesos para a Wavelet automaticamente (self.weight = nn.Parameter(kernel)).

In [ ]:
# Camada Wavelet 
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    # Passamos os canais de entrada
    def __init__(self, in_channels):  
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        kernel_base = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)
        
        # Formato (out, in, h, w) - para wavelet, out = in
        kernel = kernel_base.repeat(in_channels, 1, 1, 1)
        
        # Transformamos em nn.Parameter para ser TREINÁVEL
        self.weight = nn.Parameter(kernel)

    def forward(self, x):
        # Como o peso já tem o tamanho correto, basta aplicar a convolução
        # O parâmetro chamado "groups" garante que cada canal seja processado isoladamente
        return F.conv2d(x, self.weight, stride=2, groups=x.shape[1])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    # Passamos os canais de entrada
    def __init__(self, in_channels):  
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        kernel_base = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)
        
        # Formato (out, in, h, w) - para wavelet, out = in
        kernel = kernel_base.repeat(in_channels, 1, 1, 1)
        
        # Transformamos em nn.Parameter para ser TREINÁVEL
        self.weight = nn.Parameter(kernel)

    def forward(self, x):
        # Como o peso já tem o tamanho correto, basta aplicar a convolução
        # O parâmetro chamado "groups" garante que cada canal seja processado isoladamente
        return F.conv2d(x, self.weight, stride=2, groups=x.shape[1])


## 3. Modelo com 3 convoluções de CNN sem camada Wavelet e com Data Augmentation

### Remoção da camada Wavelet

A ideia da remoção da cadama wavelet no processo do modelo é para validar se seria ela que está travando o resultando entre 60-65%.

Se nesse teste tivermos uma acurácia maior de fato a camada Wavelet estava "cegando" a rede. O filtro de média (0.5) está eliminando as frequências que distinguem suas classes. 

Teremos que mudar a Wavelet de Haar para algo mais complexo ou inicializar os pesos de forma diferente.

In [5]:
"""
MODELO COM 3 CONVOLUÇÕES E REMOVENDO A WAVELET PARA TESTE
"""
import torch
import torch.nn as nn

class TripleCNN(nn.Module):
    def __init__(self):
        super(TripleCNN, self).__init__()
                
        # BLOCO 1: Recebe 1 canal e sai 32
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Substitui Wavelet1
        
        # BLOCO 2: Recebe 32 e sai 64
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Substitui Wavelet2
        
        # BLOCO 3: Recebe 64 e sai 128
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Decisão Final
        # 28 -> 14 -> 7 -> 3. Logo: 128 filtros de 7x7
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = torch.relu(self.bn3(self.conv3(x)))
        
        # Vetorização e Classificação
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### 3.1 Treinando o modelo

#### Treinamento otimizado

Calcula a média de erro de todos os lotes e mostra a porcentagem de acertos enquanto o modelo aprende.

Aalém disso código já inclui o .to(device), o que acelera o treino em até 10x caso seja rodado em ambiente que tenha uma placa de vídeo ou estiver usando o Google Colab.

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Instanciar o modelo novo
model = TripleCNN()

# 2. Definir o dispositivo (GPU ou CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Definir a função de perda
criterion = nn.CrossEntropyLoss()

# 4. Definir o otimizador (Passando os parâmetros do NOVO modelo)
# o weight_decay para ajudar a chegar nos 90%
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    corretos_treino = 0
    total_treino = 0
    
    for imagens, labels in train_loader:
        # Enviar dados para o dispositivo (CPU ou GPU)
        imagens, labels = imagens.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(imagens)
        loss = criterion(outputs, labels)

        # Backward pass e otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Estatísticas de Treino
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_treino += labels.size(0)
        corretos_treino += (predicted == labels).sum().item()

    # Cálculos das médias da época
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * corretos_treino / total_treino
    
    print(f"Época [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc Treino: {epoch_acc:.2f}%")

    # Valida no test_loader a cada 5 épocas para ver se o Overfitting (Sobreajuste) parou
    if (epoch + 1) % 5 == 0:
        model.eval()
        corretos_teste = 0
        total_teste = 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                _, pred = torch.max(out, 1)
                total_teste += lbls.size(0)
                corretos_teste += (pred == lbls).sum().item()
        print(f">>> Acurácia de Teste: {100 * corretos_teste / total_teste:.2f}%")

Época [1/20] - Loss: 1.7305 - Acc Treino: 50.67%
Época [2/20] - Loss: 0.9225 - Acc Treino: 49.33%
Época [3/20] - Loss: 0.7633 - Acc Treino: 58.30%
Época [4/20] - Loss: 0.7192 - Acc Treino: 56.95%
Época [5/20] - Loss: 0.6603 - Acc Treino: 60.54%
>>> Acurácia de Teste: 62.89%
Época [6/20] - Loss: 0.6591 - Acc Treino: 62.33%
Época [7/20] - Loss: 0.6680 - Acc Treino: 56.95%
Época [8/20] - Loss: 0.6561 - Acc Treino: 62.33%
Época [9/20] - Loss: 0.6516 - Acc Treino: 64.13%
Época [10/20] - Loss: 0.6349 - Acc Treino: 61.43%
>>> Acurácia de Teste: 64.95%
Época [11/20] - Loss: 0.6656 - Acc Treino: 57.40%
Época [12/20] - Loss: 0.6450 - Acc Treino: 65.02%
Época [13/20] - Loss: 0.6425 - Acc Treino: 62.33%
Época [14/20] - Loss: 0.6461 - Acc Treino: 64.13%
Época [15/20] - Loss: 0.6501 - Acc Treino: 61.88%
>>> Acurácia de Teste: 64.95%
Época [16/20] - Loss: 0.6584 - Acc Treino: 62.78%
Época [17/20] - Loss: 0.6412 - Acc Treino: 63.68%
Época [18/20] - Loss: 0.6365 - Acc Treino: 63.23%
Época [19/20] - Los

### 3.2 Testando o modelo

In [8]:
model.eval()
corretos = 0
total = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    for imagens, labels in test_loader:
        
        imagens, labels = imagens.to(device), labels.to(device)

        outputs = model(imagens)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        corretos += (predicted == labels).sum().item()

print(f"Acurácia: {100 * corretos / total:.2f}%")

Acurácia: 64.95%


## 4. Resultado do teste

Dado que a acurácia se manteve praticamente a mesma comparanda com o modelo com Wavelet, então entende-se que não seria a camada Wavelet o problema do nosso resultado ruim. 

O que está acontecendo é que o modelo (seja com Wavelet ou com Pool) não está conseguindo encontrar padrões que diferenciem as duas classes.